# 00 · Data Acquisition & Cleaning

## Part 0 — Acquire & Clean

Two real-world datasets, two **disguised-missingness** traps that a naive `isna()` never catches:

| dataset | role | the trap |
|---|---|---|
| **Telco Customer Churn** | cross-sectional Advanced EDA | `TotalCharges` is stored as *text*; 11 blanks → NaN, all at `tenure == 0` → **MAR** |
| **Shiller S&P 500** (monthly, 1871→) | time-series half | `0` is a *placeholder*; PE10/CPI can never be 0 |

Goal: produce model-ready frames in `data/processed/`, with the cleaning decisions made explicit.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = pathlib.Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import data, eda
eda.set_style()
pd.set_option("display.width", 120, "display.max_columns", 30)
print("setup ok | numpy", np.__version__, "| pandas", pd.__version__)


setup ok | numpy 2.1.3 | pandas 2.3.3


### 1. Telco — the text-column trap
`TotalCharges` looks complete, but it is `object` (text). For a money column, that alone is a red flag.

In [2]:
raw = data.load_telco_raw()
print("shape:", raw.shape)
print("TotalCharges dtype:", raw.TotalCharges.dtype, "| isna() says:", int(raw.TotalCharges.isna().sum()), "missing")
coerced = pd.to_numeric(raw.TotalCharges, errors="coerce")
bad = coerced.isna()
print("after to_numeric -> blanks exposed:", int(bad.sum()))
print("tenure of those rows:", sorted(raw.loc[bad, "tenure"].unique().tolist()), "  <-- all brand-new customers (MAR)")

shape: (7043, 21)
TotalCharges dtype: object | isna() says: 0 missing
after to_numeric -> blanks exposed: 11
tenure of those rows: [0]   <-- all brand-new customers (MAR)


The missingness is **not random** — it is mechanically tied to `tenure == 0` (a customer who has never completed a billing cycle). That is **MAR**. The honest fix is `TotalCharges = 0` for those rows. `clean_telco()` also recasts `SeniorCitizen` (0/1 → Yes/No), drops `customerID`, and adds a numeric `churn_flag`.

In [3]:
telco = data.clean_telco()
print("clean shape:", telco.shape, "| remaining NaN in TotalCharges:", int(telco.TotalCharges.isna().sum()))
print("churn rate: %.1f%%  (imbalanced, like real churn)" % (100 * telco.churn_flag.mean()))
telco.dtypes.to_frame("dtype").T

clean shape: (7043, 21) | remaining NaN in TotalCharges: 0
churn rate: 26.5%  (imbalanced, like real churn)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_flag
dtype,category,category,category,category,int64,category,category,category,category,category,category,category,category,category,category,category,category,float64,float64,category,int64


### 2. Shiller — the zero-as-placeholder trap
`isna()` reports nothing, yet a CAPE ratio or CPI of **0** is impossible.

In [4]:
sraw = data.load_shiller_raw()
zero_counts = (sraw == 0).sum()
print("columns with literal-zero placeholders:")
print(zero_counts[zero_counts > 0].to_string())
print("\nPE10 leading zeros = CAPE needs 10y trailing earnings, so it only starts in 1881.")
print("Trailing zeros on the fundamentals = recent months not yet reported (price stays fresh).")

columns with literal-zero placeholders:
Dividend                 35
Earnings                 35
Consumer Price Index     32
Long Interest Rate       32
Real Price               32
Real Dividend            35
Real Earnings            35
PE10                    152

PE10 leading zeros = CAPE needs 10y trailing earnings, so it only starts in 1881.
Trailing zeros on the fundamentals = recent months not yet reported (price stays fresh).


`clean_shiller()` converts those zeros to NaN (only where 0 is impossible), sets a monthly `DatetimeIndex`, and derives the features we actually analyse: `return`, `log_return`, `real_return`, `cape`, `cape_z`.

In [5]:
shiller = data.clean_shiller()
print("range:", shiller.index.min().date(), "->", shiller.index.max().date(), "| freq:", shiller.index.freq)
print("PE10 NaN after fix:", int(shiller.PE10.isna().sum()), "(were zeros)")
shiller[["SP500", "Earnings", "Consumer Price Index", "return", "log_return", "cape", "cape_z"]].tail(3)

range: 1871-01-01 -> 2026-05-01 | freq: <MonthBegin>
PE10 NaN after fix: 152 (were zeros)


,SP500,Earnings,Consumer Price Index,return,log_return,cape,cape_z
Date,,,,,,,
2026-03-01,6654.42,NaN,NaN,-0.034725,-0.035343,NaN,NaN
2026-04-01,6957.01,NaN,NaN,0.045472,0.044468,NaN,NaN
2026-05-01,7412.55,NaN,NaN,0.065479,0.063425,NaN,NaN


### 3. VIX (companion, for Part 4)

In [6]:
vix = data.clean_vix()
print("VIX:", vix.shape, "|", vix.index.min().date(), "->", vix.index.max().date())
vix.tail(2)

VIX: (9210, 4) | 1990-01-02 -> 2026-06-18


,open,high,low,close
date,,,,
2026-06-17,16.08,18.84,16.02,18.44
2026-06-18,17.23,17.60,16.35,16.40


### 4. Persist everything to `data/processed/`

In [7]:
frames = data.build_processed()
for k, v in frames.items():
    print(f"{k:8s} {str(v.shape):12s} -> data/processed/")
print("\nPart 0 complete. Cleaned, documented, reproducible.")

telco    (7043, 21)   -> data/processed/


shiller  (1865, 14)   -> data/processed/
vix      (9210, 4)    -> data/processed/

Part 0 complete. Cleaned, documented, reproducible.
